<a href="https://colab.research.google.com/github/wubbleramble/DaSiWa-TrainFlow/blob/main/Anima_trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌟 Anima Lora Trainer

❗ **Colab Premium is not needed** but recommended for larger datasets. Ideally you'd be changing the runtime to an A100 and use the maximum batch size.  



This colab is based on the work of [Kohya-ss](https://github.com/kohya-ss/sd-scripts) and [Hollowstrawberry](https://github.com/hollowstrawberry/kohya-colab). Thank you!

### ⭕ Disclaimer
The purpose of this document is to research bleeding-edge technologies in the field of machine learning inference.  
Please read and follow the [Google Colab guidelines](https://research.google.com/colaboratory/faq.html) and its [Terms of Service](https://research.google.com/colaboratory/tos_v3.html).

| |GitHub|🇬🇧 English|🇪🇸 Spanish|
|:--|:-:|:-:|:-:|
| 🏠 **Homepage** | [![GitHub](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/github.svg)](https://github.com/uYouUs/kohya-colab) | | |
| ⭐ **Lora Trainer** | [![GitHub](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/github.svg)](https://github.com/uYouUs/kohya-colab/blob/main/Lora_Trainer.ipynb) | [![Open in Colab](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/colab-badge.svg)](https://colab.research.google.com/github/uYouUs/kohya-colab/blob/main/Lora_Trainer.ipynb) | [![Abrir en Colab](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/colab-badge-spanish.svg)](https://colab.research.google.com/github/uYouUs/kohya-colab/blob/main/Spanish_Lora_Trainer.ipynb) |
| 🌟 **Simple XL Trainer** | [![GitHub](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/github.svg)](https://github.com/uYouUs/kohya-colab/blob/main/Simple_XL_Trainer.ipynb) | [![Open in Colab](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/colab-badge.svg)](https://colab.research.google.com/github/uYouUs/kohya-colab/blob/main/Simple_XL_Trainer.ipynb) | [![Abrir en Colab](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/colab-badge-spanish.svg)](https://colab.research.google.com/github/uYouUs/kohya-colab/blob/main/Spanish_Simple_XL_Trainer.ipynb) |
| 🌟 **XL Lora Trainer** | [![GitHub](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/github.svg)](https://github.com/uYouUs/kohya-colab/blob/main/Lora_Trainer_XL.ipynb) | [![Open in Colab](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/colab-badge.svg)](https://colab.research.google.com/github/uYouUs/kohya-colab/blob/main/Lora_Trainer_XL.ipynb) | [![Abrir en Colab](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/colab-badge-spanish.svg)](https://colab.research.google.com/github/uYouUs/kohya-colab/blob/main/Spanish_Lora_Trainer_XL.ipynb) |
| 🌟 **Advanced XL Trainer** | [![GitHub](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/github.svg)](https://github.com/uYouUs/kohya-colab/blob/main/Advanced_Trainer_XL.ipynb) | [![Open in Colab](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/colab-badge.svg)](https://colab.research.google.com/github/uYouUs/kohya-colab/blob/main/Advanced_Trainer_XL.ipynb) | [![Abrir en Colab](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/colab-badge-spanish.svg)](https://colab.research.google.com/github/uYouUs/kohya-colab/blob/main/Spanish_Advanced_Trainer_XL.ipynb) |
| 🌟 **Anima Lora Trainer** | [![GitHub](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/github.svg)](https://github.com/uYouUs/kohya-colab/blob/main/Lora_Trainer_Anima.ipynb) | [![Open in Colab](https://raw.githubusercontent.com/uYouUs/kohya-colab/main/assets/colab-badge.svg)](https://colab.research.google.com/github/uYouUs/kohya-colab/blob/main/Lora_Trainer_Anima.ipynb) |  |

In [ ]:
#@title Delete .npz cache files
#@markdown Optional: type a folder here BEFORE running to set it. Leave empty to reuse the last saved folder.
#@markdown You can also change the folder in the "Folder:" box AFTER running; it is saved automatically.
FOLDER_OVERRIDE = "" #@param {type:"string"}

# ===== Imports =====
import os, glob, json, threading
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive

# ===== Google Drive Mount (skipped if already mounted) =====
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# ===== Settings (remembered between runs, even after the runtime resets) =====
DEFAULT_PATH = "/content/drive/MyDrive/Loras/w11-f1t-tr4in3r-illustrious/dataset"
SETTINGS_FILE = "/content/drive/MyDrive/.npz_cleaner_settings.json"

def load_settings():
    try:
        with open(SETTINGS_FILE) as f:
            return json.load(f)
    except Exception:
        return {}

def save_settings(data):
    try:
        with open(SETTINGS_FILE, "w") as f:
            json.dump(data, f)
    except Exception as e:
        print("Could not save settings:", e)

# Which folder to start with:
#   1) a NEW value typed in FOLDER_OVERRIDE (different from the last one used)
#   2) otherwise the last folder saved from the "Folder:" box
#   3) otherwise DEFAULT_PATH
settings = load_settings()
start_path = settings.get("path", DEFAULT_PATH)
form_value = FOLDER_OVERRIDE.strip()
if form_value and form_value != settings.get("last_form_value"):
    start_path = form_value
settings["last_form_value"] = form_value
settings["path"] = start_path
save_settings(settings)

# ===== UI Layout =====
path_input = widgets.Text(
    value=start_path,
    description="Folder:",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "60px"}
)

delete_button = widgets.Button(
    description="Delete .npz cache files",
    button_style="danger",
    icon="trash",
    layout=widgets.Layout(width="220px")
)

output = widgets.Output()

# Save the folder shortly after typing stops (avoids writing to Drive on every keystroke)
_timer = None

def _save_path():
    s = load_settings()
    s["path"] = path_input.value.strip()
    save_settings(s)

def on_path_change(change):
    global _timer
    if _timer:
        _timer.cancel()
    _timer = threading.Timer(1.0, _save_path)
    _timer.start()

path_input.observe(on_path_change, names="value")

PROTECTED = {"/", "/content", "/content/drive", "/content/drive/MyDrive"}

def on_delete_clicked(b):
    with output:
        clear_output()
        folder = path_input.value.strip().rstrip("/") or "/"
        if _timer:
            _timer.cancel()
        _save_path()  # make sure the folder in the box is the one remembered

        if folder in PROTECTED:
            print(f"Refused: '{folder}' is too broad. Pick your dataset folder.")
            return
        if not os.path.isdir(folder):
            print(f"Folder not found: {folder}")
            return

        cache_files = glob.glob(os.path.join(folder, "**/*.npz"), recursive=True)
        deleted, failed = 0, 0
        for f in cache_files:
            try:
                os.remove(f)
                deleted += 1
            except Exception as e:
                failed += 1
                print(f"Could not delete {f}: {e}")
        print(f"Deleted {deleted} cache files" + (f" ({failed} failed)" if failed else ""))

delete_button.on_click(on_delete_clicked)

display(widgets.VBox([path_input, delete_button, output]))

In [ ]:
#@markdown ## Repetition Calculator ⌛📝
#@markdown Calculates the number of repetitions to use to train your Lora. Remember that in `SDXL and Pony` a batch of `4` is used.
#@markdown If you use Colab Pro, calculate your repetitions with `8` batch size
# Define the Variables
# Number of images
num_images = 100 # @param{type:"number"}
# Number of repetitions
num_repeats = 2 # @param{type:"number"}
# Number of epochs
num_epochs = 10 # @param{type:"number"}
# Batch size
batch_size = 4 # @param{type:"number"}

# Calculate the result
resultado = (num_images * num_repeats * num_epochs) / batch_size

# Display the result
print("\33[96mThe total number of repetitions is:\033[0m", resultado)


The total number of repetitions is: 1697.5


In [ ]:
#@markdown ### 📂 Unzip dataset
#@markdown It's much slower to upload individual files to your Drive, so you may want to upload a zip if you have your dataset in your computer.
zip = "/content/drive/MyDrive/Loras/project-name/dataset.zip" #@param {type:"string"}
extract_to = "/content/drive/MyDrive/Loras/project-name/" #@param {type:"string"}

import os, zipfile

if not os.path.exists('/content/drive'):
  from google.colab import drive
  print("📂 Connecting to Google Drive...")
  drive.mount('/content/drive')

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip, 'r') as f:
  f.extractall(extract_to)

print("✅ Done")


In [ ]:
import os
import re
import toml
import pathlib
import threading
import zipfile
import time
import subprocess
from IPython.display import Markdown, display

# These carry information from past executions
if "old_model_url" not in globals():
  old_model_url = None
if "dependencies_installed" not in globals():
  dependencies_installed = False
if "model_file" not in globals():
  model_file = None
if "model_cached" not in globals():
  model_cached = False
if "diffusers_model" not in globals():
  diffusers_model = None

# These may be set by other cells, some are legacy
if "custom_dataset" not in globals():
  custom_dataset = None
if "override_dataset_config_file" not in globals():
  override_dataset_config_file = None
if "continue_from_lora" not in globals():
  continue_from_lora = ""
if "override_config_file" not in globals():
  override_config_file = None

COLAB = True
SOURCE = "https://github.com/uYouUs/sd-scripts"
BRANCH = "dev"
# Reference LoRA was trained on upstream commit b1011100ffc996d6c545bc042f3fa6331f740289.
# Setting COMMIT to that hash only works if the fork above contains it; otherwise git prints an error.
COMMIT = None
BETTER_EPOCH_NAMES = True
LOAD_TRUNCATED_IMAGES = True
try:
  LOWRAM = int(next(line.split()[1] for line in open('/proc/meminfo') if "MemTotal" in line)) / (1024**2) < 15
except:
  LOWRAM = False

#@title ## 🚩 Start Here

#@markdown ### 💤 Auto Off
#@markdown *If this option is checked, the environment will automatically disconnect once training is complete.*
auto_off = True  # @param {type: "boolean"}
if auto_off:
    print("\033[96mAuto Off turned on\033[0m")
else:
    print("\033[96mAuto Off turned off\033[0m")

#@markdown ### ▶️ Setup
#@markdown Your project name will be the same as the folder containing your images. Spaces aren't allowed.
project_name = "New Project" #@param {type:"string"}
project_name = project_name.strip()

#@markdown ### 📂 Dataset
#@markdown Your dataset should already be placed in your Google Drive, inside a folder named after your project.
#@markdown Expected path: `MyDrive/Loras/<project_name>/dataset`
folder_structure = "Organize by project (MyDrive/Loras/project_name/dataset)" #@param ["Organize by project (MyDrive/Loras/project_name/dataset)"]

#@markdown Decide the model that will be downloaded and used for training.
training_model = "Anima Base v1.0" #@param ["Anima Base v1.0", "Anima Preview", "Anima Preview2", "Anima Preview3"]
optional_custom_training_model = "" #param {type:"string"}
#@markdown Using a diffusers model utilizes less resources. The options above will work with or without diffusers.
load_diffusers = True #param {type:"boolean"}
force_dif = False if optional_custom_training_model and not load_diffusers else True # Built in models will force use diffuser versions as they handle caching between swapping models
#@markdown Civitai API token for models that require login.
civitoken = "" #@param {type:"string"}
#@markdown Use wandb if you want to visualize the progress of your training over time.
wandb_key = "" #@param {type:"string"}
model_url = None
if optional_custom_training_model:
  model_url = optional_custom_training_model
  model_url = model_url.strip()
  model_file = "/content/custom_model.safetensors"
elif "Anima Base v1.0" in training_model:
  anima_model = "base-v1.0"
elif "Anima Preview3" in training_model:
  anima_model = "preview3-base"
elif "Anima Preview2" in training_model:
  anima_model ="preview2"
else:
  anima_model = "preview"


#@markdown ### ▶️ Processing
resolution = 1024 #param {type:"slider", min:768, max:1536, step:128}
caption_extension = ".txt" #@param [".txt", ".caption"]
# @markdown The activation tag at the start of each caption is kept (keep_tokens=1).<p>
shuffle_tags = False #@param {type:"boolean"}
activation_tags = "1" #@param [0,1,2,3]
keep_tokens = int(activation_tags)

# @markdown
num_repeats = 1 #@param {type:"number"}
preferred_unit = "Epochs" #@param ["Epochs", "Steps"]
how_many = 70 #@param {type:"number"}
max_train_epochs = how_many if preferred_unit == "Epochs" else None
max_train_steps = how_many if preferred_unit == "Steps" else None
#@markdown Saving more epochs will let you compare your Lora's progress better.
save_every_n_epochs = 1 #@param {type:"number"}
keep_only_last_n_epochs = 70 #@param {type:"number"}
if not save_every_n_epochs:
  save_every_n_epochs = max_train_epochs
if not keep_only_last_n_epochs:
  keep_only_last_n_epochs = max_train_epochs

# @markdown The text encoder is frozen in Anima, so its rate stays 0.
unet_lr = 7.5e-5 #@param {type:"number"}
text_encoder_lr = 0 #param {type:"number"}
lr_scheduler = "cosine" #@param ["constant", "cosine", "cosine_with_restarts", "constant_with_warmup", "linear", "polynomial"]
lr_scheduler_number = 3 #@param {type:"number"}
lr_warmup_ratio = 0.05 #@param {type:"slider", min:0.0, max:0.2, step:0.01}
lr_warmup_steps = 0
# @markdown
min_snr_gamma = 0 #@param {type:"slider", min:0.0, max:16.0, step:0.5}
multinoise = False #@param {type:"boolean"}

#@markdown ### ▶️ Structure
#@markdown RECIPE: plain LoRA, dim 32, alpha 32. The conv values below only apply if LoCon is selected.
lora_type = "LoRA" #@param ["LoRA", "LoCon"]
network_dim = 32 #@param {type:"slider", min:1, max:64, step:1}
network_alpha = 32 #@param {type:"slider", min:1, max:64, step:1}
conv_dim = 8 #@param {type:"slider", min:1, max:64, step:1}
conv_alpha = 4 #@param {type:"slider", min:1, max:64, step:1}

network_module = "networks.lora_anima"
network_args = None
if lora_type.lower() == "locon":
  network_args = [f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}"]

# @markdown Note: caption dropout (0.05) needs the text encoder outputs NOT cached, so that cache stays off.
train_batch_size = 8 #@param {type:"slider", min:1, max:16, step:1}
cross_attention = "torch" #@param ["torch", "flash"]
mixed_precision = "bf16" #@param ["bf16", "full_fp16"]
cache_latents = True #@param {type:"boolean"}
cache_latents_to_drive = True #@param {type:"boolean"}
cache_text_encoder_outputs  = False  #param {type:"boolean"}
caption_dropout_rate = 0.05

#@markdown ### ▶️ Advanced
#@markdown RECIPE: AdamW8bit with weight_decay=0.01 (default betas).
optimizer = "AdamW8bit" #@param ["AdamW8bit", "Prodigy", "DAdaptation", "DadaptAdam", "DadaptLion", "AdamW", "Lion", "SGDNesterov", "SGDNesterov8bit", "AdaFactor"]
optimizer_args = "weight_decay=0.01" #@param {type:"string"}
optimizer_args = [a.strip() for a in optimizer_args.split(' ') if a]
recommended_values = False #@param {type:"boolean"}

if any(opt in optimizer.lower() for opt in ["dadapt", "prodigy"]):
  if recommended_values:
    unet_lr = 1
    network_alpha = network_dim

lr_scheduler_num_cycles = lr_scheduler_number if lr_scheduler == "cosine_with_restarts" else 0
lr_scheduler_power = lr_scheduler_number if lr_scheduler == "polynomial" else 0

#@markdown ### ▶️ Generate Sample Image
#@markdown Leave empty to skip sample images (the reference run had none).
sample_prompt = "" #@param {type:"string"}

#@markdown ### ▶️ Ready
#@markdown You can now run this cell to cook your Lora. Good luck! <p>


# 👩‍💻 Cool code goes here

root_dir = "/content" if COLAB else pathlib.Path.home() / "Loras"
deps_dir = os.path.join(root_dir, "deps")
repo_dir = os.path.join(root_dir, "kohya-trainer")
model_dir = os.path.join(repo_dir, "model")

# Path handling: dataset and outputs are always read directly from Google Drive,
# organized under MyDrive/Loras/<project_name>/, matching the "organize by project" layout.
main_dir      = os.path.join(root_dir, "drive/MyDrive/Loras")
log_folder    = os.path.join(main_dir, "_logs")
config_folder = os.path.join(main_dir, project_name)
images_folder = os.path.join(main_dir, project_name, "dataset")
output_folder = os.path.join(main_dir, project_name, "output")

config_file = os.path.join(config_folder, "training_config.toml")
dataset_config_file = os.path.join(config_folder, "dataset_config.toml")
accelerate_config_file = os.path.join(repo_dir, "accelerate_config/config.yaml")


anima = "anima-"+anima_model+".safetensors"
if not optional_custom_training_model:
  model_file = os.path.join(model_dir, "split_files/diffusion_models/"+anima)
qwen3_file = os.path.join(model_dir, "split_files/text_encoders/qwen_3_06b_base.safetensors")
vae_file = os.path.join(model_dir, "split_files/vae/qwen_image_vae.safetensors")

def install_dependencies():
  print("Cloning Kohya")
  os.chdir(root_dir)
  !git clone {SOURCE} {repo_dir}
  os.chdir(repo_dir)
  if BRANCH:
    !git checkout {BRANCH}
  if COMMIT:
    !git reset --hard {COMMIT}
  !wget https://raw.githubusercontent.com/uYouUs/kohya-colab/main/train_network_xl_wrapper.py -q -O train_network_xl_wrapper.py
  !wget https://raw.githubusercontent.com//uYouUs/kohya-colab/main/dracula.py -q -O dracula.py

  !pip install uv
  !uv pip install diffusers==0.36.0 transformers==4.57.6 bitsandbytes==0.49.1 pytorch-lightning==1.9.0 voluptuous==0.13.1 \
    invisible-watermark==0.2.0 prodigyopt==1.0.0 \
    dadaptation==3.1 lion-pytorch==0.1.2 ftfy==6.1.1 --no-progress
  !uv pip install -e . --no-progress
  if cross_attention == "flash":
    !uv pip install -q flash-attn

  # patch kohya for minor stuff
  if LOWRAM:
    !sed -i "s@cpu@cuda@" library/model_util.py
  if LOAD_TRUNCATED_IMAGES:
    !sed -i 's/from PIL import Image/from PIL import Image, ImageFile\nImageFile.LOAD_TRUNCATED_IMAGES=True/g' library/train_util.py # fix truncated jpegs error
  if BETTER_EPOCH_NAMES:
    !sed -i 's/{:06d}/{:02d}/g' library/train_util.py # make epoch names shorter
    !sed -i 's/"." + args.save_model_as)/"-{:02d}.".format(num_train_epochs) + args.save_model_as)/g' train_network.py # name of the last epoch will match the rest

  from accelerate.utils import write_basic_config
  if not os.path.exists(accelerate_config_file):
    write_basic_config(save_location=accelerate_config_file)

  os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
  os.environ["BITSANDBYTES_NOWELCOME"] = "1"
  os.environ["SAFETENSORS_FAST_GPU"] = "1"

def get_supported_recipe_args():
  # The reference LoRA used these Anima-specific settings. They are only added if this
  # version of the training script actually offers them, so a wrong name can never crash the run.
  wanted = {
    "timestep_sample_method": "logit_normal",
    "discrete_flow_shift": 3.0,
    "sigmoid_scale": 1.0,
  }
  try:
    help_text = subprocess.run(
      ["python", "anima_train_network.py", "--help"],
      cwd=repo_dir, capture_output=True, text=True, timeout=300
    )
    help_text = help_text.stdout + help_text.stderr
  except Exception as e:
    print(f"⚠️ Could not read the script's options ({e}). Skipping timestep settings.")
    return {}
  found = {}
  for key, value in wanted.items():
    if f"--{key}" in help_text:
      found[key] = value
      print(f"✅ Script supports --{key}, using {value}")
    else:
      print(f"ℹ️ Script has no --{key}, leaving it at the script default")
  return found

def validate_dataset():
  global lr_warmup_steps, lr_warmup_ratio, caption_extension, keep_tokens, model_url
  supported_types = (".png", ".jpg", ".jpeg", ".webp", ".bmp")

  print("\n💿 Checking dataset...")
  if not project_name.strip() or any(c in project_name for c in " .()\"'\\/"):
    print("💥 Error: Please choose a valid project name.")
    return

  # Find the folders and files
  if custom_dataset:
    try:
      datconf = toml.loads(custom_dataset)
      datasets = [d for d in datconf["datasets"][0]["subsets"]]
    except:
      print(f"💥 Error: Your custom dataset is invalid or contains an error! Please check the original template.")
      return
    reg = [d.get("image_dir") for d in datasets if d.get("is_reg", False)]
    datasets_dict = {d["image_dir"]: d["num_repeats"] for d in datasets}
    folders = datasets_dict.keys()
    files = [f for folder in folders for f in os.listdir(folder)]
    images_repeats = {folder: (len([f for f in os.listdir(folder) if f.lower().endswith(supported_types)]), datasets_dict[folder]) for folder in folders}
  else:
    reg = []
    folders = [images_folder]
    if not os.path.exists(images_folder):
      print(f"💥 Error: The folder {images_folder.replace('/content/drive/', '')} doesn't exist. "
            f"Make sure your dataset is placed at MyDrive/Loras/{project_name}/dataset")
      return
    files = os.listdir(images_folder)
    images_repeats = {images_folder: (len([f for f in files if f.lower().endswith(supported_types)]), num_repeats)}

  # Validation
  for folder in folders:
    if not os.path.exists(folder):
      print(f"💥 Error: The folder {folder.replace('/content/drive/', '')} doesn't exist.")
      return
  for folder, (img, rep) in images_repeats.items():
    if not img:
      print(f"💥 Error: Your {folder.replace('/content/drive/', '')} folder is empty.")
      return
  test_files = []
  for f in files:
    if not f.lower().endswith((caption_extension, ".npz")) and not f.lower().endswith(supported_types):
      print(f"💥 Error: Invalid file in dataset: \"{f}\". Aborting.")
      return
    for ff in test_files:
      if f.endswith(supported_types) and ff.endswith(supported_types) \
          and os.path.splitext(f)[0] == os.path.splitext(ff)[0]:
        print(f"💥 Error: The files {f} and {ff} cannot have the same name. Aborting.")
        return
    test_files.append(f)

  if caption_extension and not [txt for txt in files if txt.lower().endswith(caption_extension)]:
    caption_extension = ""
  if continue_from_lora and not (continue_from_lora.endswith(".safetensors") and os.path.exists(continue_from_lora)):
    print(f"💥 Error: Invalid path to existing Lora. Example: /content/drive/MyDrive/Loras/example.safetensors")
    return

  # Pretty stuff

  pre_steps_per_epoch = sum(img*rep for (img, rep) in images_repeats.values())
  steps_per_epoch = pre_steps_per_epoch/train_batch_size
  total_steps = max_train_steps or int(max_train_epochs*steps_per_epoch)
  estimated_epochs = int(total_steps/steps_per_epoch)
  lr_warmup_steps = int(total_steps*lr_warmup_ratio)

  for folder, (img, rep) in images_repeats.items():
    print("📁"+folder.replace("/content/drive/", "") + (" (Regularization)" if folder in reg else ""))
    print(f"📈 Found {img} images with {rep} repeats, equaling {img*rep} steps.")
  print(f"📉 Divide {pre_steps_per_epoch} steps by {train_batch_size} batch size to get {steps_per_epoch} steps per epoch.")
  if max_train_epochs:
    print(f"🔮 There will be {max_train_epochs} epochs, for around {total_steps} total training steps.")
  else:
    print(f"🔮 There will be {total_steps} steps, divided into {estimated_epochs} epochs and then some.")
  print(f"🔥 Warmup steps: {lr_warmup_steps}")
  print("🎯 Reference recipe check: 116 images, 232 steps/epoch, 30 epochs, 6960 total steps, 348 warmup steps.")

  if total_steps > 10000:
    print("💥 Error: Your total steps are too high. You probably made a mistake. Aborting...")
    return

  return True

def create_config():
  global dataset_config_file, config_file, model_file

  if override_config_file:
    config_file = override_config_file
    print(f"\n⭕ Using custom config file {config_file}")
  else:
    recipe_args = get_supported_recipe_args()

    config_dict = {
      "network_arguments": {
        "unet_lr": unet_lr,
        "text_encoder_lr": 0,
        "network_dim": network_dim,
        "network_alpha": network_alpha,
        "network_module": network_module,
        "network_args": network_args,
        "network_train_unet_only": text_encoder_lr == 0 or cache_text_encoder_outputs,
        "network_weights": continue_from_lora if continue_from_lora else None
      },
      "optimizer_arguments": {
        "learning_rate": unet_lr,
        "lr_scheduler": lr_scheduler,
        "lr_scheduler_num_cycles": lr_scheduler_num_cycles if lr_scheduler == "cosine_with_restarts" else None,
        "lr_scheduler_power": lr_scheduler_power if lr_scheduler == "polynomial" else None,
        "lr_warmup_steps": lr_warmup_steps if lr_scheduler != "constant" else None,
        "optimizer_type": optimizer,
        "optimizer_args": optimizer_args if optimizer_args else None,
      },
      "training_arguments": {
        "pretrained_model_name_or_path": model_file,
        "qwen3": qwen3_file,
        "vae": vae_file,
        "max_train_steps": max_train_steps,
        "max_train_epochs": max_train_epochs,
        "train_batch_size": train_batch_size,
        "seed": 42,
        # RECIPE: no "max_token_length" line on purpose, so long captions are not cut by an SD-era token limit.
        "min_snr_gamma": min_snr_gamma if min_snr_gamma > 0 else None,
        "lowram": LOWRAM,
        "no_half_vae": True,
        "gradient_checkpointing": True,
        "gradient_accumulation_steps": 1,
        "max_grad_norm": 1.0,
        "max_data_loader_n_workers": 3,
        "persistent_data_loader_workers": True,
        "mixed_precision": "fp16" if mixed_precision == "full_fp16" else mixed_precision,
        "full_bf16": False,  # RECIPE: reference metadata shows mixed_precision=bf16 only
        "full_fp16": mixed_precision == "full_fp16",
        "cache_latents": cache_latents,
        "cache_latents_to_disk": cache_latents_to_drive,
        "cache_text_encoder_outputs": cache_text_encoder_outputs,
        "min_timestep": 0,
        "max_timestep": 1000,
        "prior_loss_weight": 1.0,
        "multires_noise_iterations": 6 if multinoise else None,
        "multires_noise_discount": 0.3 if multinoise else None,
        **recipe_args,
      },
      "saving_arguments": {
        "save_precision": "bf16",
        "save_model_as": "safetensors",
        "save_every_n_epochs": save_every_n_epochs,
        "save_last_n_epochs": keep_only_last_n_epochs,
        "output_name": project_name,
        "output_dir": output_folder,
        "log_prefix": project_name,
        "logging_dir": log_folder,
        "wandb_api_key": wandb_key if wandb_key else None,
        "log_with": "wandb" if wandb_key else None,
        "sample_prompts": prompt_file if sample_prompt else None,
        "sample_every_n_epochs": save_every_n_epochs if sample_prompt else None,
        "sample_sampler": "euler_a" if sample_prompt else None,
      }
    }

    for key in config_dict:
      if isinstance(config_dict[key], dict):
        config_dict[key] = {k: v for k, v in config_dict[key].items() if v is not None}

    with open(config_file, "w") as f:
      f.write(toml.dumps(config_dict))
    print(f"\n📄 Config saved to {config_file}")

  if override_dataset_config_file:
    dataset_config_file = override_dataset_config_file
    print(f"⭕ Using custom dataset config file {dataset_config_file}")
  else:
    dataset_config_dict = {
      "general": {
        "resolution": resolution,
        "shuffle_caption": shuffle_tags and not cache_text_encoder_outputs,
        "keep_tokens": keep_tokens,
        "flip_aug": False,
        "caption_extension": caption_extension,
        "caption_dropout_rate": caption_dropout_rate,
        "enable_bucket": True,
        "bucket_no_upscale": False,
        "bucket_reso_steps": 64,
        "min_bucket_reso": 256,
        "max_bucket_reso": 4096,  # kept high on purpose (reference used 1024)
      },
      "datasets": toml.loads(custom_dataset)["datasets"] if custom_dataset else [
        {
          "subsets": [
            {
              "num_repeats": num_repeats,
              "image_dir": images_folder,
              "class_tokens": None if caption_extension else project_name
            }
          ]
        }
      ]
    }

    for key in dataset_config_dict:
      if isinstance(dataset_config_dict[key], dict):
        dataset_config_dict[key] = {k: v for k, v in dataset_config_dict[key].items() if v is not None}

    with open(dataset_config_file, "w") as f:
      f.write(toml.dumps(dataset_config_dict))
    print(f"📄 Dataset config saved to {dataset_config_file}")

def download_model():
  global old_model_url, model_url, model_file
  real_model_url = model_url  # Keep model url intact to compare against old model url to detect a change.

  # Test for google drive model
  if real_model_url.startswith("/content/drive/"):
    model_file = real_model_url
    if model_file.lower().endswith(".safetensors") or model_file.lower().endswith(".ckpt"):
      print(f"\nTraing will be done with drive model at {model_file}, check it is valid if you recieve errors\n")
      return True
    return False

  else:
    # Define local filename
    if not model_file or not model_file.endswith((".ckpt", ".safetensors")):
      if real_model_url.lower().endswith((".ckpt", ".safetensors")):
        model_file = f"/content{real_model_url[real_model_url.rfind('/'):]}"
      else:
        model_file = "/content/downloaded_model.safetensors"
        if os.path.exists(model_file):
          !rm "{model_file}"

    # HuggingFace
    if m := re.search(r"(?:https?://)?(?:www\.)?huggingface\.co/[^/]+/[^/]+/blob", real_model_url):
      real_model_url = real_model_url.replace("blob", "resolve")
    # Civitai
    elif m := re.search(r"(?:https?://)?(?:www\\.)?civitai\.com/models/([0-9]+)(/[A-Za-z0-9-_]+)?", real_model_url):
      if m.group(2):
        model_file = f"/content{m.group(2)}.safetensors"
      if m := re.search(r"modelVersionId=([0-9]+)", real_model_url):
        real_model_url = f"https://civitai.com/api/download/models/{m.group(1)}"
      else:
        raise ValueError("💥 optional_custom_training_model contains a civitai link, but the link doesn't include a modelVersionId. You can also right click the download button to copy the direct download link.")
      if civitoken:
        real_model_url = real_model_url+"?token="+civitoken
    # Download checkpoint
    !apt -y install aria2 -qq
    !aria2c "{real_model_url}" --console-log-level=warn -c -s 16 -x 16 -k 10M -d / -o "{model_file}"

    # Validation

    if model_file.lower().endswith(".safetensors"):
      from safetensors.torch import load_file as load_safetensors
      try:
        test = load_safetensors(model_file)
        del test
      except:
        new_model_file = os.path.splitext(model_file)[0]+".ckpt"
        !mv "{model_file}" "{new_model_file}"
        model_file = new_model_file
        print(f"Renamed model to {os.path.splitext(model_file)[0]}.ckpt")

    if model_file.lower().endswith(".ckpt"):
      from torch import load as load_ckpt
      try:
        test = load_ckpt(model_file)
        del test
      except:
        return False

  return True

def getModel():
  global diffusers_model, model_url, model_file, old_model_url
  from huggingface_hub import snapshot_download
  from huggingface_hub.utils import disable_progress_bars
  disable_progress_bars() # Disable model download progress bars to clean up output from threading mess
  if not optional_custom_training_model:
    snapshot_download(repo_id="circlestone-labs/Anima", allow_patterns=["*qwen_image_vae.safetensors", "*qwen_3_06b_base.safetensors", f"*{anima}"], local_dir=f"{repo_dir}/model")
  elif old_model_url != model_url or not model_file or not os.path.exists(model_file):
    print("\n🔄 Getting model...")
    if not download_model():
      print("\n💥 Error: The model you specified is invalid or corrupted."
            "\nIf you're using an URL, please check that the model is accessible without being logged in."
            "\nYou can try civitai or huggingface URLs, or a path in your Google Drive starting with /content/drive/MyDrive")
      return
    snapshot_download(repo_id="circlestone-labs/Anima", allow_patterns=["*qwen_image_vae.safetensors", "*qwen_3_06b_base.safetensors"], local_dir=f"{repo_dir}/model")
    print()
  else:
    print("\n🔄 Model already downloaded.\n")
  old_model_url = model_url # only update old model url if we successfully downloaded it to prevent not downloading model if it previously failed on a change.

def generateSample(): # Generate required files for sample generation
  global output_folder, sample_prompt, prompt_file
  if sample_prompt:
    prompt_file = os.path.abspath(os.path.join(output_folder, "sample"))
    if not os.path.exists(prompt_file): # Make sample directory so prompt file creation does not fail.
      os.mkdir(prompt_file)
    prompt_file = os.path.join(prompt_file, "prompt.txt")
    with open(prompt_file, "w") as f:
      f.write(sample_prompt)

def thInstallDep():
  global dependencies_installed
  print("\n🏭 Installing dependencies...\n")
  t0 = time.time()
  install_dependencies()
  t1 = time.time()
  dependencies_installed = True
  print(f"\n✅ Installation finished in {int(t1-t0)} seconds. \nWaiting on Model...\n")

def main():
  global dependencies_installed
  threadList = []
  # Download model with a thread
  thr1 = threading.Thread(target = getModel)
  threadList.append(thr1)
  thr1.start()

  if not dependencies_installed: # Begin installing dependencies while user gives gdrive permissions
    thr2 = threading.Thread(target = thInstallDep)
    threadList.append(thr2)
    thr2.start()
  else:
    print("\n✅ Dependencies already installed.")

  if COLAB and not os.path.exists('/content/drive'): # Request drive access to read the dataset directly from it
    from google.colab import drive
    print("\n📂 Connecting to Google Drive...")
    drive.mount('/content/drive')

  for th in threadList: # Make sure model is done before continuing
    th.join()

  for dir in (main_dir, deps_dir, repo_dir, log_folder, output_folder, config_folder):
    os.makedirs(dir, exist_ok=True)

  if not validate_dataset():
    return

  generateSample() # Make sure sample file exists prior to creating config file
  create_config()

  print("\n⭐ Starting trainer...\n")
  os.chdir(repo_dir)

  # RECIPE: --num_processes=1 (one GPU on Colab, matches reference batch/step maths)
  !accelerate launch --quiet --config_file={accelerate_config_file} --num_cpu_threads_per_process=2 --num_processes=1 anima_train_network.py --dataset_config={dataset_config_file} --config_file={config_file} --attn_mode={cross_attention}

  if not get_ipython().__dict__['user_ns']['_exit_code']:
    display(Markdown("### ✅ Done! [Go download your Lora from Google Drive](https://drive.google.com/drive/my-drive)\n"
                     "### There will be several files, you should try the latest version (the file with the largest number next to it)"))

main()

# Auto Off: disconnect the Colab runtime once training finishes
if auto_off:
    print("\033[96mAuto Off is on, the environment will disconnect in 30 seconds\033[0m")
    from google.colab import runtime
    time.sleep(30)
    runtime.unassign()
else:
    print("\033[96mAuto Off is disabled, the environment will not disconnect\033[0m")

## *️⃣ Extras

You can run these before starting the training.

In [ ]:
#@markdown ### Continue

#@markdown Here you can write a path in your Google Drive to load an existing Lora file to continue training on.<p>
#@markdown **Warning:** It's not the same as one long training session. The epochs start from scratch, and it may have worse results.
continue_from_lora = "" #@param {type:"string"}
if continue_from_lora and not continue_from_lora.startswith("/content/drive/MyDrive"):
  import os
  continue_from_lora = os.path.join("/content/drive/MyDrive", continue_from_lora)


### 📚 Multiple folders in dataset
Below is a template allowing you to define multiple folders in your dataset. You must include the location of each folder and you can set different number of repeats for each one. To add more folders simply copy and paste the sections starting with `[[datasets.subsets]]`.

When enabling this, the number of repeats set in the main cell will be ignored, and the main folder set by the project name will also be ignored.

You can make one of them a regularization folder by adding `is_reg = true`  
You can also set different `keep_tokens`, `flip_aug`, etc.

When using google drive for datasets, the default format is:
`content/drive/MyDrive/Loras/example/dataset/normal_images`

For in-drive unzipping if dataset, the default location is:
`content/Loras/example/dataset/normal_images`


In [ ]:
custom_dataset = """
[[datasets]]

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/good_images"
num_repeats = 3

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/normal_images"
num_repeats = 1

"""

In [ ]:
custom_dataset = None

In [ ]:
#@markdown ### 🔢 Count datasets
#@markdown Google Drive makes it impossible to count the files in a folder, so this will show you the file counts in all folders and subfolders.
folder = "/content/drive/MyDrive/Loras" #@param {type:"string"}

import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("📂 Connecting to Google Drive...\n")
    drive.mount('/content/drive')

tree = {}
exclude = ("_logs", "/output")
for i, (root, dirs, files) in enumerate(os.walk(folder, topdown=True)):
  dirs[:] = [d for d in dirs if all(ex not in d for ex in exclude)]
  images = len([f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))])
  captions = len([f for f in files if f.lower().endswith(".txt")])
  others = len(files) - images - captions
  path = root[folder.rfind("/")+1:]
  tree[path] = None if not images else f"{images:>4} images | {captions:>4} captions |"
  if tree[path] and others:
    tree[path] += f" {others:>4} other files"

pad = max(len(k) for k in tree)
print("\n".join(f"📁{k.ljust(pad)} | {v}" for k, v in tree.items() if v))


# 📈 Plot training results
You can do this after running the trainer. You don't need this unless you know what you're doing.  
The first cell below may fail to load all your logs. Keep trying the second cell until all data has loaded.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir={log_folder}/

In [ ]:
from tensorboard import notebook
notebook.display(port=6006, height=800)